In [4]:
!pip install torch-geometric -q

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm, global_add_pool, global_max_pool
from torch_geometric.loader import DataLoader
from torch.cuda.amp import autocast, GradScaler
import os
import time
import numpy as np
from tqdm import tqdm
import random
import gc
from pathlib import Path

In [3]:
DATA_DIR = "/kaggle/input/datasets/jyotisharma16/400k-dataset/400k_dataset"

TRAIN_PATH = os.path.join(DATA_DIR, "train.pt")
VAL_PATH = os.path.join(DATA_DIR, "val.pt")
TEST_PATH = os.path.join(DATA_DIR, "test.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import os
print(os.listdir("/kaggle/input/datasets/jyotisharma16/400k-dataset/400k_dataset"))

['train.pt', 'val.pt', 'test.pt']


In [10]:
print("Loading dataset...")

train_data = torch.load(TRAIN_PATH, weights_only=False)
val_data = torch.load(VAL_PATH, weights_only=False)
test_data = torch.load(TEST_PATH, weights_only=False)

train_loader = DataLoader(train_data, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=2)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")


Loading dataset...
Train: 350000 | Val: 25000 | Test: 25000


In [13]:
# MODEL

class AdvancedGINE(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()

        self.node_encoder = nn.Sequential(
            nn.Linear(9, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.edge_encoder = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for _ in range(6):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(BatchNorm(hidden_dim))

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)

        add_pool = global_add_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        out = torch.cat([add_pool, max_pool], dim=1)
        return self.fc(out).squeeze()





In [14]:
# TRAIN FUNCTION

def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        with autocast():
            pred = model(batch)
            loss = F.l1_loss(pred, batch.y.squeeze())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


In [15]:
# EVAL FUNCTION

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch)

            preds.extend(pred.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(np.abs(preds - targets))
    return mae

In [17]:
# TRAIN LOOP

model = AdvancedGINE().to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = GradScaler()

EPOCHS = 50

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}")

    train_loss = train_epoch(model, train_loader, optimizer, scaler)
    val_mae = evaluate(model, val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val MAE: {val_mae:.4f}")



# FINAL TEST

test_mae = evaluate(model, test_loader)
print(f"\nFinal Test MAE: {test_mae:.4f}")

/tmp/ipykernel_55/2979399322.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Epoch 1


  0%|          | 0/1368 [00:00<?, ?it/s]/tmp/ipykernel_55/2663040528.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 1368/1368 [00:30<00:00, 45.10it/s]


Train Loss: 0.3161
Val MAE: 0.4792

Epoch 2


100%|██████████| 1368/1368 [00:30<00:00, 44.46it/s]


...

Train Loss: 0.0913
Val MAE: 0.0994

Epoch 49


100%|██████████| 1368/1368 [00:31<00:00, 43.81it/s]


Train Loss: 0.0903
Val MAE: 0.1006

Epoch 50


100%|██████████| 1368/1368 [00:30<00:00, 44.34it/s]


Train Loss: 0.0899
Val MAE: 0.1000

Final Test MAE: 0.0982


In [6]:
BASE_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/final_dataset1"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_TOTAL = 1_400_000   # STOP CONDITION


# NORMALIZE → LIST
def normalize_to_list(data):
    if isinstance(data, (list, tuple)):
        return list(data)
    if hasattr(data, "to_data_list"):
        return data.to_data_list()
    return [data]


# VALID GRAPH CHECK
def is_valid_graph(data):
    try:
        if not hasattr(data, 'x') or data.x is None or data.x.size(0) == 0:
            return False

        if not hasattr(data, 'edge_index') or data.edge_index is None or data.edge_index.size(1) == 0:
            return False

        if not hasattr(data, 'y') or data.y is None:
            return False

        y = data.y

        if torch.is_tensor(y):
            if torch.isnan(y).any() or torch.isinf(y).any():
                return False
        else:
            if y is None or (isinstance(y, float) and np.isnan(y)):
                return False

        return True

    except:
        return False


# VALID DICT CHECK
def is_valid_dict(d):
    try:
        t = d.get("target", None)

        if t is None:
            return False

        if isinstance(t, float) and np.isnan(t):
            return False

        return True

    except:
        return False


# FIND FILES
print("Searching dataset files...")

all_files = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.endswith(".pt"):
            path = os.path.join(root, file)

            # skip 400k dataset
            if "400k" in path.lower():
                continue

            all_files.append(path)

print(f"Found {len(all_files)} files")


# LOAD + FILTER (WITH EARLY STOP)
all_data = []

skipped_graphs = 0
nan_skipped = 0
corrupted_files = 0

for idx, file_path in enumerate(all_files, 1):

    print(f"\nLoading {idx}/{len(all_files)}")

    # STOP when 2.4M reached
    if len(all_data) >= TARGET_TOTAL:
        print("Reached 1.4M clean samples → stopping loading")
        break

    try:
        data = torch.load(file_path, map_location="cpu", weights_only=False)
    except Exception:
        print(f"Corrupted: {file_path}")
        corrupted_files += 1
        continue

    data_list = normalize_to_list(data)

    valid_count = 0

    for g in data_list:

        # STOP INSIDE LOOP ALSO (VERY IMPORTANT)
        if len(all_data) >= TARGET_TOTAL:
            break

        try:
            # PyG graph
            if hasattr(g, "x"):
                if is_valid_graph(g):
                    all_data.append(g)
                    valid_count += 1
                else:
                    skipped_graphs += 1

            # dict dataset
            elif isinstance(g, dict):
                if is_valid_dict(g):
                    all_data.append(g)
                    valid_count += 1
                else:
                    nan_skipped += 1

            else:
                skipped_graphs += 1

        except:
            skipped_graphs += 1

    print(f"Added: {valid_count} | Total so far: {len(all_data):,}")

    del data
    del data_list
    gc.collect()


# SUMMARY
print("\nDATA SUMMARY")
print(f"Final dataset size: {len(all_data):,}")
print(f"Invalid graphs: {skipped_graphs:,}")
print(f"NaN skipped: {nan_skipped:,}")
print(f"Corrupted files: {corrupted_files:,}")


# SAFETY CHECK
if len(all_data) < 1000:
    raise ValueError("Too little data, something is wrong")


# SHUFFLE
print("\nShuffling dataset...")
random.seed(42)
random.shuffle(all_data)

Searching dataset files...
Found 24 files

Loading 1/24
Added: 437500 | Total so far: 437,500

Loading 2/24
Added: 31250 | Total so far: 468,750

Loading 3/24
Added: 31250 | Total so far: 500,000

Loading 4/24
Added: 437500 | Total so far: 937,500

Loading 5/24
Added: 31250 | Total so far: 968,750

Loading 6/24
Added: 31250 | Total so far: 1,000,000

Loading 7/24
Added: 283343 | Total so far: 1,283,343

Loading 8/24
Added: 20238 | Total so far: 1,303,581

Loading 9/24
Added: 20240 | Total so far: 1,323,821

Loading 10/24
Added: 76179 | Total so far: 1,400,000

Loading 11/24
Reached 2.4M clean samples → stopping loading

DATA SUMMARY
Final dataset size: 1,400,000
Invalid graphs: 0
NaN skipped: 0
Corrupted files: 0

Shuffling dataset...


In [7]:
# SPLIT
train_size = 1_000_000
val_size = 200_000
test_size = 200_000

train_data = all_data[:train_size]
val_data = all_data[train_size:train_size + val_size]
test_data = all_data[train_size + val_size:]


print("\nFINAL SPLIT")
print(f"Train: {len(train_data):,}")
print(f"Val:   {len(val_data):,}")
print(f"Test:  {len(test_data):,}")
print(f"Total: {len(all_data):,}")


# SAVE
print("\nSaving dataset...")

torch.save(train_data, os.path.join(OUTPUT_DIR, "train.pt"))
torch.save(val_data, os.path.join(OUTPUT_DIR, "val.pt"))
torch.save(test_data, os.path.join(OUTPUT_DIR, "test.pt"))

print("\nDONE")
print(f"Saved at: {OUTPUT_DIR}")


FINAL SPLIT
Train: 1,000,000
Val:   200,000
Test:  200,000
Total: 1,400,000

Saving dataset...

DONE
Saved at: /kaggle/working/final_dataset1


In [8]:
!kaggle datasets init -p /kaggle/working/final_dataset1

Data package template written to: /kaggle/working/final_dataset1/dataset-metadata.json


In [9]:
!kaggle datasets create -p /kaggle/working/final_dataset1

Default slug detected, please change values before uploading


In [10]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU ")
# PATHS
DATA_DIR = "/kaggle/working/final_dataset1"

TRAIN_PATH = f"{DATA_DIR}/train.pt"
VAL_PATH   = f"{DATA_DIR}/val.pt"
TEST_PATH  = f"{DATA_DIR}/test.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATA
print(" Loading dataset...")

train_data = torch.load(TRAIN_PATH, weights_only=False)
val_data   = torch.load(VAL_PATH, weights_only=False)
test_data  = torch.load(TEST_PATH, weights_only=False)

train_loader = DataLoader(train_data, batch_size=256, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=2)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")


CUDA Available: True
GPU Name: Tesla T4
 Loading dataset...
Train: 1000000 | Val: 200000 | Test: 200000


In [11]:
# MODEL

class AdvancedGINE(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()

        self.node_encoder = nn.Sequential(
            nn.Linear(9, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.edge_encoder = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for _ in range(6):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(BatchNorm(hidden_dim))

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)

        add_pool = global_add_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        out = torch.cat([add_pool, max_pool], dim=1)
        return self.fc(out).squeeze()



In [12]:
# TRAIN FUNCTION

def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, leave=False):
        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        
        with autocast(device_type='cuda'):
            pred = model(batch)
            loss = F.l1_loss(pred, batch.y.squeeze())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


In [4]:
# EVAL FUNCTION

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch)

            preds.extend(pred.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(np.abs(preds - targets))
    return mae



In [19]:
model = AdvancedGINE().to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = GradScaler('cuda')

EPOCHS = 50
best_val = float("inf")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}")

    model.train()   # ensure training mode

    train_loss = train_epoch(model, train_loader, optimizer, scaler)
    
    model.eval()    # ensure eval mode
    val_mae = evaluate(model, val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val MAE: {val_mae:.4f}")

    # Save best model
    if val_mae < best_val:
        best_val = val_mae
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
        print("Best model saved!")


Epoch 1


Train Loss: 0.2606
Val MAE: 0.2194
Best model saved!

Epoch 2


Train Loss: 0.1857
Val MAE: 0.1769
Best model saved!

Epoch 3


Train Loss: 0.1656
Val MAE: 0.1576
Best model saved!

Epoch 4


Train Loss: 0.1534
Val MAE: 0.1562
Best model saved!

Epoch 5


...

Train Loss: 0.0970
Val MAE: 0.1025

Epoch 47


Train Loss: 0.0969
Val MAE: 0.1075

Epoch 48


Train Loss: 0.0967
Val MAE: 0.1040

Epoch 49


Train Loss: 0.0966
Val MAE: 0.1014
Best model saved!

Epoch 50


Train Loss: 0.0961
Val MAE: 0.1018


In [20]:
# FINAL TEST

model.load_state_dict(torch.load("/kaggle/working/best_model.pt"))

test_mae = evaluate(model, test_loader)
print(f"\n Final Test MAE: {test_mae:.4f}")


 Final Test MAE: 0.1011
